In [0]:
from pyspark.sql import functions as f
import sys
sys.path.append('..')
sys.path.append('../..')

import lib_etl.validations_ETL as validations
from lib.job_manager import load_config, split_config

In [0]:
%run ../../config/utils

In [0]:
config = load_config(etl_config_path)
data_paths, club_square_config, config_validation = split_config(config)
run_as_date = dbutils.widgets.get("run_as_date")

### Load variables

### Transform 

In [0]:
df_transaction_detail = spark.sql(f"""
    SELECT 
        cast(PURCH_HDR_ID as long) as PURCH_HDR_ID
        ,cast(PURCH_DTL_ID as int) as PURCH_DTL_ID
        ,cast(PURCH_DT as date) as PURCH_DT
        ,GTIN_CD
        ,ARTICLE_NBR
        ,MC_CD
        ,cast(
            CASE WHEN DISCOUNT_TYPE_CD = 'ZECM' AND SALES_CTGRY_CD = '03'
            THEN 0.0
            ELSE EXTENDED_PRC_AMT END
            as double
        ) as EXTENDED_PRC_AMT
        ,cast(EXTENDED_UNIT_PRC_AMT as double) as EXTENDED_UNIT_PRC_AMT
        ,cast(SALES_QTY as double) as SALES_QTY
        ,SALES_UOM
        ,cast(QTY_IN_UNITS as int) as QTY_IN_UNITS
        ,cast(NORMAL_PRC_AMT as double) as NORMAL_PRC_AMT
        ,cast(NORMAL_UNIT_PRC_AMT as double) as NORMAL_UNIT_PRC_AMT
        ,cast(REDUCTION_AMT as double) as REDUCTION_AMT
        ,SCANNED_VS_KEYED_IND
        ,DISCOUNT_TYPE_CD
        ,cast(DISCOUNT_PURCH_DTL_ID as int) as DISCOUNT_PURCH_DTL_ID
        ,VOIDED_FLAG
        ,VOIDED_PURCH_DTL_ID
        ,RSN_CD
        ,SALES_CTGRY_CD
        ,RETURN_IND
        ,REBATE_IND
        ,OFFER_ID as VECTOR_OFFER_ID
    FROM 
        {bronze_transaction_detail}
    WHERE 
        DISCOUNT_PURCH_DTL_ID IS DISTINCT FROM 'PURCH_DTL_ID'
""").filter((f.col('PURCH_DT') >= "2016-01-01")).dropDuplicates()

df_transaction_detail.createOrReplaceTempView("source")

In [0]:
validations.validate_table(
        spark, "intermediate", 'detail', config_validation, df_transaction_detail, stats_etl_path
    )

### Merge

In [0]:
df_transaction_detail.write.mode("overwrite").saveAsTable(silver_transaction_detail)

if archive_flag:
    save_archive(df_transaction_detail, silver_transaction_detail_archive, run_as_date)